# 06 — Deterministic and Monte Carlo timing chase

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
import random

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
)

# ============================================================
# Load inputs
# ============================================================

profiles = (
    spark.table(
        table_name(
            config,
            "gold_historical_schema",
            "observed_driver_profiles"
        )
    )
    .filter("support_status = 'sufficient'")
    .select("driver_id", "pooled_median_pace")
    .collect()
)

virtual = (
    spark.table(
        table_name(
            config,
            "gold_scenario_schema",
            "virtual_max_profile"
        )
    )
    .first()
)

variability = [
    float(row.lap_deviation)
    for row in (
        spark.table(
            table_name(
                config,
                "gold_historical_schema",
                "historical_variability"
            )
        )
        .select("lap_deviation")
        .collect()
    )
]

if not profiles or not variability:
    raise ValueError(
        "Historical Gold inputs are required before simulation"
    )

race_laps = 30
vmax = float(virtual.virtual_max_pace_seconds)

# ============================================================
# Center historical variability
# ============================================================

variability_mean = (
    sum(variability) / len(variability)
)

centered_variability = [
    value - variability_mean
    for value in variability
]

centered_mean = (
    sum(centered_variability)
    / len(centered_variability)
)

print("Supported drivers:", len(profiles))
print(
    "Historical variability observations:",
    len(variability)
)
print(
    "Original variability mean:",
    variability_mean
)
print(
    "Centered variability mean:",
    centered_mean
)

# ============================================================
# Deterministic timing catch
# ============================================================

def maximum_delay(field, virtual_pace):
    return min(
        race_laps * (
            float(row.pooled_median_pace)
            - virtual_pace
        )
        for row in field
    )

deterministic_delay = float(
    maximum_delay(
        profiles,
        vmax
    )
)

print(
    "\nTiming catch only; "
    "not a physical overtake model."
)

print(
    "Deterministic max delay:",
    deterministic_delay
)

# ============================================================
# Monte Carlo setup
#
# Common random numbers:
# generate the historical field scenarios once, then evaluate
# every possible starting delay over exactly the same scenarios.
# ============================================================

SIMULATIONS = 5000
SEED = 42

random.seed(SEED)

scenario_full_field_margins = []

vmax_zero_delay_total = (
    race_laps * vmax
)

for _ in range(SIMULATIONS):

    margins = []

    for row in profiles:

        base_pace = float(
            row.pooled_median_pace
        )

        driver_total = sum(
            base_pace
            + random.choice(
                centered_variability
            )
            for _ in range(race_laps)
        )

        margin = (
            driver_total
            - vmax_zero_delay_total
        )

        margins.append(margin)

    scenario_full_field_margins.append(
        min(margins)
    )

# ============================================================
# Catch probability function
# ============================================================

def full_catch_probability(delay):

    return sum(
        margin >= float(delay)
        for margin
        in scenario_full_field_margins
    ) / SIMULATIONS

# ============================================================
# Search range
#
# Negative delay:
# Virtual Max receives a head start.
#
# Positive delay:
# Virtual Max starts later.
# ============================================================

min_delay = -10.0

max_delay = max(
    10.0,
    deterministic_delay * 2
)

step = 0.1

n_steps = int(
    round(
        (max_delay - min_delay)
        / step
    )
)

candidates = [
    round(
        min_delay + i * step,
        4
    )
    for i in range(
        n_steps + 1
    )
]

curve = [
    (
        float(delay),
        float(
            full_catch_probability(
                delay
            )
        )
    )
    for delay in candidates
]

# ============================================================
# Probability thresholds
#
# Maximum delay compatible with at least:
# 50%, 80%, 90% full-field timing catch probability.
# ============================================================

thresholds = {
    p: max(
        (
            delay
            for delay, probability
            in curve
            if probability >= p
        ),
        default=None
    )
    for p in (
        0.5,
        0.8,
        0.9
    )
}

probability_zero = float(
    full_catch_probability(0.0)
)

probability_deterministic = float(
    full_catch_probability(
        deterministic_delay
    )
)

print("\nMonte Carlo thresholds:")

print(
    "P50:",
    thresholds[0.5]
)

print(
    "P80:",
    thresholds[0.8]
)

print(
    "P90:",
    thresholds[0.9]
)

print(
    "\nProbability at zero delay:",
    probability_zero
)

print(
    "Probability at deterministic delay:",
    probability_deterministic
)

# ============================================================
# Selected probability curve
# ============================================================

print(
    "\nSelected Monte Carlo "
    "probability curve:"
)

for delay, probability in curve:

    is_integer_delay = (
        abs(
            delay - round(delay)
        ) < 1e-9
    )

    is_near_deterministic = (
        abs(
            delay - deterministic_delay
        ) <= 0.06
    )

    if (
        is_integer_delay
        or is_near_deterministic
    ):
        print(
            f"delay={delay:6.2f} s   "
            f"full-field catch "
            f"probability={probability:.3f}"
        )

# ============================================================
# Validation
# ============================================================

if all(
    thresholds[p] is not None
    for p in (
        0.5,
        0.8,
        0.9
    )
):
    assert (
        thresholds[0.9]
        <= thresholds[0.8]
        <= thresholds[0.5]
    ), (
        "Probability thresholds "
        "have invalid ordering."
    )

probabilities = [
    probability
    for _, probability in curve
]

assert all(
    probabilities[i]
    >= probabilities[i + 1]
    for i in range(
        len(probabilities) - 1
    )
), (
    "Monte Carlo probability curve "
    "is unexpectedly non-monotonic."
)

# ============================================================
# Build summary table
# ============================================================

summary_schema = StructType([
    StructField(
        "scenario_scope",
        StringType(),
        False
    ),
    StructField(
        "supported_field_size",
        IntegerType(),
        False
    ),
    StructField(
        "race_laps",
        IntegerType(),
        False
    ),
    StructField(
        "simulations",
        IntegerType(),
        False
    ),
    StructField(
        "deterministic_max_delay_seconds",
        DoubleType(),
        False
    ),
    StructField(
        "probability_at_zero_delay",
        DoubleType(),
        False
    ),
    StructField(
        "delay_50pct",
        DoubleType(),
        True
    ),
    StructField(
        "delay_80pct",
        DoubleType(),
        True
    ),
    StructField(
        "delay_90pct",
        DoubleType(),
        True
    ),
    StructField(
        "caveat",
        StringType(),
        False
    ),
])

summary = spark.createDataFrame(
    [(
        "observed_field",
        int(len(profiles)),
        int(race_laps),
        int(SIMULATIONS),
        float(
            deterministic_delay
        ),
        float(
            probability_zero
        ),
        (
            None
            if thresholds[0.5]
            is None
            else float(
                thresholds[0.5]
            )
        ),
        (
            None
            if thresholds[0.8]
            is None
            else float(
                thresholds[0.8]
            )
        ),
        (
            None
            if thresholds[0.9]
            is None
            else float(
                thresholds[0.9]
            )
        ),
        (
            "SYNTHETIC WHAT-IF; "
            "timing catch only; "
            "historical within-driver "
            "deviations centered to zero; "
            "Virtual Max held deterministic "
            "because no observed Virtual Max "
            "variability exists; "
            "common random numbers used "
            "across delays"
        ),
    )],
    schema=summary_schema,
)

# ============================================================
# Build probability curve table
# ============================================================

curve_schema = StructType([
    StructField(
        "delay_seconds",
        DoubleType(),
        False
    ),
    StructField(
        "full_field_catch_probability",
        DoubleType(),
        False
    ),
])

curve_df = spark.createDataFrame(
    [
        (
            float(delay),
            float(probability)
        )
        for delay, probability
        in curve
    ],
    schema=curve_schema,
)

# ============================================================
# Publish Gold Scenario
#
# We deliberately drop/recreate these two tables because their
# schema evolved during the POC.
# ============================================================

summary_table = table_name(
    config,
    "gold_scenario_schema",
    "monte_carlo_scenario_summary"
)

curve_table = table_name(
    config,
    "gold_scenario_schema",
    "monte_carlo_probability_curve"
)

spark.sql(
    f"DROP TABLE IF EXISTS {summary_table}"
)

spark.sql(
    f"DROP TABLE IF EXISTS {curve_table}"
)

summary.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        summary_table
    )

curve_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option(
        "overwriteSchema",
        "true"
    ) \
    .saveAsTable(
        curve_table
    )

# ============================================================
# Final output
# ============================================================

print(
    "\nGold Scenario tables published:"
)

print(summary_table)
print(curve_table)

summary.show(
    truncate=False
)

In [ ]:
print("Number of supported drivers:", len(profiles))
print("Number of variability observations:", len(variability))

print("\nVariability:")
print("min:", min(variability))
print("max:", max(variability))
print("mean:", sum(variability) / len(variability))

sorted_var = sorted(variability)
print("median:", sorted_var[len(sorted_var) // 2])

print("\nMonte Carlo probability curve:")
for delay, probability in curve:
    print(
        f"delay={delay:6.3f} s  "
        f"full-field catch probability={probability:.3f}"
    )

print(
    "\nMaximum probability observed:",
    max(probability for _, probability in curve)
)

print(
    "Probability at zero delay:",
    curve[0][1]
)